# Web2Media Service - Google Colab / Local Jupyter

Notebook n?y t? d?ng l?i Web2Media Python/FastAPI service cho API t?o video.

Ch?y c?c cell theo th? t?. Tr?n Google Colab, cell server s? in link proxy `Docs URL`. N?u ch?y local tr?n Windows/Jupyter, setup cell s? b? qua `apt-get`, c?i package b?ng ??ng Python kernel, v? server m? t?i `http://127.0.0.1:4526/docs`. `.env` kh?ng ???c nh?ng v?o notebook ?? tr?nh l? secret.


In [ ]:
# Recreate the Web2Media project files inside the runtime.
from pathlib import Path

TEXT_FILES = {'requirements.txt': 'fastapi>=0.111,<1.0\nuvicorn[standard]>=0.30,<1.0\nplaywright>=1.45,<2.0\nhttpx>=0.27,<1.0\n', 'requirements-dev.txt': '-r requirements.txt\npytest>=8.0,<9.0\npytest-asyncio>=0.23,<1.0\n', '.gitignore': '.env\ntemp/\n__pycache__/\n.pytest_cache/\n.venv/\n*.pyc\n', '.dockerignore': '.git\n.env\ntemp\n__pycache__\n.pytest_cache\n.venv\n*.pyc\n', 'Dockerfile': 'FROM python:3.12-slim\n\nENV PYTHONDONTWRITEBYTECODE=1\nENV PYTHONUNBUFFERED=1\nENV PORT=4526\n\nWORKDIR /app\n\nRUN apt-get update && apt-get install -y --no-install-recommends \\\n    ffmpeg \\\n    curl \\\n    && rm -rf /var/lib/apt/lists/*\n\nCOPY requirements.txt ./\nRUN pip install --no-cache-dir -r requirements.txt\nRUN python -m playwright install --with-deps chromium\n\nCOPY . .\n\nEXPOSE 4526\n\nHEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\\n    CMD python -c "import sys, urllib.request; sys.exit(0 if urllib.request.urlopen(\'http://localhost:4526/api/health\', timeout=5).status == 200 else 1)"\n\nCMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT:-4526}"]\n', 'docker-compose.yml': 'services:\n  web2media-service:\n    build: .\n    image: web2media-service:latest\n    ports:\n      - "4526:4526"\n    restart: unless-stopped\n    environment:\n      - PYTHON_ENV=production\n    env_file:\n      - .env\n    networks:\n      internal_shared:\n        aliases:\n          - web2media-service \n      default: {}\n\nnetworks:\n  internal_shared:\n    external: true\n    name: internal_shared\n', 'app/__init__.py': '"""Web2Media Python service package."""\n', 'app/config.py': 'from __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\n\nBASE_DIR = Path(__file__).resolve().parents[1]\n\n\ndef _env_int(name: str, default: int) -> int:\n    value = os.getenv(name)\n    if not value:\n        return default\n    try:\n        return int(value)\n    except ValueError:\n        return default\n\n\nBG_PRESETS = [\n    {"index": 0, "label": "Rừng", "gradient": "radial-gradient(ellipse at 30% 60%, #0d2b14 0%, #050e08 60%, #020608 100%)"},\n    {"index": 1, "label": "Đêm", "gradient": "radial-gradient(ellipse at 50% 100%, #0d1535 0%, #040810 60%, #010205 100%)"},\n    {"index": 2, "label": "Hoàng hôn", "gradient": "linear-gradient(170deg, #0d0510 0%, #2a0e20 40%, #0a0508 100%)"},\n    {"index": 3, "label": "Ao hồ", "gradient": "radial-gradient(ellipse at 50% 80%, #071e2e 0%, #030c14 50%, #010408 100%)"},\n    {"index": 4, "label": "Núi", "gradient": "linear-gradient(160deg, #070a14 0%, #111828 40%, #050710 100%)"},\n    {"index": 5, "label": "Lúa", "gradient": "radial-gradient(ellipse at 50% 90%, #1a2208 0%, #0a0e04 60%, #040602 100%)"},\n    {"index": 6, "label": "Biển", "gradient": "linear-gradient(180deg, #03060e 0%, #061224 40%, #040d1a 100%)"},\n    {"index": 7, "label": "Tím", "gradient": "radial-gradient(ellipse at 50% 100%, #1e0f35 0%, #08051a 60%, #03020c 100%)"},\n]\n\nCOLOR_PRESETS = [\n    {"index": 0, "name": "Xanh lá", "h": 105, "s": 85, "l": 65, "hex": "#5fdf47"},\n    {"index": 1, "name": "Vàng", "h": 55, "s": 95, "l": 68, "hex": "#f5e94a"},\n    {"index": 2, "name": "Xanh lam", "h": 195, "s": 90, "l": 65, "hex": "#4dc8f0"},\n    {"index": 3, "name": "Cam", "h": 35, "s": 95, "l": 65, "hex": "#f5b44a"},\n    {"index": 4, "name": "Trắng", "h": 0, "s": 0, "l": 90, "hex": "#e6e6e6"},\n    {"index": 5, "name": "Hồng", "h": 320, "s": 80, "l": 75, "hex": "#ef8ad6"},\n]\n\nDIRECTIONS = ["up", "down", "left", "right", "up-left", "up-right", "down-left", "down-right", "random"]\nGLOW_LEVELS = ["low", "mid", "high"]\n\nDEFAULT_CONFIG = {\n    "count": 80,\n    "size": 2.5,\n    "speed": 1.0,\n    "colorMode": "preset",\n    "colorIndex": 0,\n    "customColor": "#7fff9a",\n    "glowLevel": "mid",\n    "direction": "up",\n    "spread": 0.4,\n    "bgIndex": 1,\n    "bgUrl": None,\n    "duration": 10,\n    "fps": 60,\n    "width": 1920,\n    "height": 1080,\n    "bitrate": 5_000_000,\n    "format": "webm",\n    "filename": "firefly",\n}\n\n\n@dataclass(frozen=True)\nclass ServerConfig:\n    port: int = _env_int("PORT", 3000)\n    temp_dir: Path = BASE_DIR / "temp"\n    public_dir: Path = BASE_DIR / "public"\n    max_concurrent: int = _env_int("MAX_CONCURRENT", 3)\n    max_duration: int = 120\n\n\nSERVER_CONFIG = ServerConfig()\n', 'app/openapi.py': 'from __future__ import annotations\n\nfrom typing import Any\n\nfrom fastapi import FastAPI\nfrom fastapi.openapi.utils import get_openapi\n\n\nRECORD_REQUEST_SCHEMA: dict[str, Any] = {\n    "type": "object",\n    "description": "Tham số cấu hình để tạo video. Tất cả field đều có giá trị mặc định.",\n    "properties": {\n        "count": {\n            "type": "integer",\n            "minimum": 10,\n            "maximum": 300,\n            "default": 80,\n            "example": 80,\n            "description": "Số lượng đom đóm.",\n        },\n        "size": {\n            "type": "number",\n            "minimum": 1,\n            "maximum": 6,\n            "default": 2.5,\n            "example": 2.5,\n            "description": "Kích thước mỗi đom đóm.",\n        },\n        "speed": {\n            "type": "number",\n            "minimum": 0.2,\n            "maximum": 3,\n            "default": 1.0,\n            "example": 1.0,\n            "description": "Tốc độ bay.",\n        },\n        "colorMode": {\n            "type": "string",\n            "enum": ["preset", "custom"],\n            "default": "preset",\n            "example": "preset",\n            "description": "Dùng màu preset hoặc màu hex tùy chỉnh.",\n        },\n        "colorIndex": {\n            "type": "integer",\n            "minimum": 0,\n            "maximum": 5,\n            "default": 0,\n            "example": 1,\n            "description": "Index màu preset: 0 xanh lá, 1 vàng, 2 xanh lam, 3 cam, 4 trắng, 5 hồng.",\n        },\n        "customColor": {\n            "type": "string",\n            "pattern": "^#[0-9a-fA-F]{6}$",\n            "default": "#7fff9a",\n            "example": "#ff6b9d",\n            "description": "Màu hex khi colorMode là custom.",\n        },\n        "glowLevel": {\n            "type": "string",\n            "enum": ["low", "mid", "high"],\n            "default": "mid",\n            "example": "high",\n            "description": "Cường độ phát sáng.",\n        },\n        "direction": {\n            "type": "string",\n            "enum": ["up", "down", "left", "right", "up-left", "up-right", "down-left", "down-right", "random"],\n            "default": "up",\n            "example": "random",\n            "description": "Hướng bay của đom đóm.",\n        },\n        "spread": {\n            "type": "number",\n            "minimum": 0,\n            "maximum": 1,\n            "default": 0.4,\n            "example": 0.7,\n            "description": "Độ tản mạn, 0 là bay thẳng, 1 là tản rộng.",\n        },\n        "bgIndex": {\n            "type": "integer",\n            "minimum": 0,\n            "maximum": 7,\n            "default": 1,\n            "example": 3,\n            "description": "Index background preset.",\n        },\n        "bgUrl": {\n            "type": "string",\n            "nullable": True,\n            "format": "uri",\n            "default": None,\n            "example": "https://example.com/background.jpg",\n            "description": "URL ảnh nền tùy chỉnh. Nếu có, giá trị này override bgIndex.",\n        },\n        "duration": {\n            "type": "integer",\n            "minimum": 3,\n            "maximum": 120,\n            "default": 10,\n            "example": 10,\n            "description": "Thời lượng video tính bằng giây.",\n        },\n        "fps": {\n            "type": "integer",\n            "enum": [24, 30, 60],\n            "default": 60,\n            "example": 30,\n            "description": "Số frame mỗi giây.",\n        },\n        "width": {\n            "type": "integer",\n            "minimum": 320,\n            "maximum": 3840,\n            "default": 1920,\n            "example": 1280,\n            "description": "Chiều rộng video.",\n        },\n        "height": {\n            "type": "integer",\n            "minimum": 240,\n            "maximum": 2160,\n            "default": 1080,\n            "example": 720,\n            "description": "Chiều cao video.",\n        },\n        "bitrate": {\n            "type": "integer",\n            "minimum": 1_000_000,\n            "maximum": 20_000_000,\n            "default": 5_000_000,\n            "example": 5_000_000,\n            "description": "Video bitrate theo bps.",\n        },\n        "format": {\n            "type": "string",\n            "enum": ["webm", "mp4", "gif"],\n            "default": "webm",\n            "example": "mp4",\n            "description": "Định dạng output.",\n        },\n        "filename": {\n            "type": "string",\n            "pattern": "^[a-zA-Z0-9_-]+$",\n            "maxLength": 100,\n            "default": "firefly",\n            "example": "pink-fireflies",\n            "description": "Tên file download, không bao gồm extension.",\n        },\n    },\n}\n\nRECORD_EXAMPLES: dict[str, Any] = {\n    "minimal": {\n        "summary": "Tối giản",\n        "description": "Chỉ đổi thời lượng, các tham số còn lại dùng mặc định.",\n        "value": {"duration": 5},\n    },\n    "preset_color": {\n        "summary": "Đom đóm vàng, nền rừng",\n        "value": {\n            "count": 120,\n            "size": 3,\n            "speed": 1.5,\n            "colorMode": "preset",\n            "colorIndex": 1,\n            "glowLevel": "high",\n            "bgIndex": 0,\n            "direction": "up",\n            "duration": 10,\n            "format": "webm",\n        },\n    },\n    "custom_color_mp4": {\n        "summary": "Màu tùy chỉnh + MP4",\n        "value": {\n            "count": 200,\n            "size": 2,\n            "speed": 0.8,\n            "colorMode": "custom",\n            "customColor": "#ff6b9d",\n            "glowLevel": "mid",\n            "direction": "random",\n            "spread": 0.7,\n            "bgIndex": 7,\n            "duration": 15,\n            "width": 1280,\n            "height": 720,\n            "fps": 30,\n            "format": "mp4",\n            "filename": "pink-fireflies",\n        },\n    },\n    "gif_output": {\n        "summary": "Export GIF nhẹ",\n        "value": {\n            "count": 60,\n            "size": 3.5,\n            "glowLevel": "high",\n            "colorIndex": 2,\n            "direction": "up-right",\n            "bgIndex": 3,\n            "duration": 5,\n            "width": 640,\n            "height": 360,\n            "fps": 24,\n            "format": "gif",\n            "filename": "firefly-preview",\n        },\n    },\n    "custom_background": {\n        "summary": "Ảnh nền tùy chỉnh",\n        "value": {\n            "count": 100,\n            "speed": 0.9,\n            "colorMode": "preset",\n            "colorIndex": 4,\n            "bgUrl": "https://example.com/background.jpg",\n            "direction": "random",\n            "duration": 10,\n            "format": "mp4",\n            "width": 1920,\n            "height": 1080,\n            "fps": 30,\n            "filename": "stars-fireflies",\n        },\n    },\n}\n\n\ndef build_custom_openapi(app: FastAPI) -> dict[str, Any]:\n    if app.openapi_schema:\n        return app.openapi_schema\n\n    openapi_schema = get_openapi(\n        title=app.title,\n        version=app.version,\n        description=app.description,\n        routes=app.routes,\n    )\n\n    components = openapi_schema.setdefault("components", {}).setdefault("schemas", {})\n    components["RecordRequest"] = RECORD_REQUEST_SCHEMA\n\n    record = openapi_schema["paths"]["/api/record"]["post"]\n    record.update(\n        {\n            "tags": ["Recording"],\n            "summary": "Tạo video đom đóm",\n            "description": "Render animation đom đóm với cấu hình tùy chỉnh và trả về file video.",\n            "requestBody": {\n                "required": False,\n                "content": {\n                    "application/json": {\n                        "schema": {"$ref": "#/components/schemas/RecordRequest"},\n                        "examples": RECORD_EXAMPLES,\n                    }\n                },\n            },\n            "responses": {\n                "200": {\n                    "description": "Video file được tạo thành công",\n                    "content": {\n                        "video/webm": {"schema": {"type": "string", "format": "binary"}},\n                        "video/mp4": {"schema": {"type": "string", "format": "binary"}},\n                        "image/gif": {"schema": {"type": "string", "format": "binary"}},\n                    },\n                },\n                "400": {"description": "Tham số không hợp lệ"},\n                "429": {"description": "Quá nhiều tiến trình đồng thời"},\n                "500": {"description": "Lỗi server"},\n            },\n        }\n    )\n\n    openapi_schema["paths"]["/api/presets"]["get"].update(\n        {"tags": ["Presets"], "summary": "Danh sách preset có sẵn"}\n    )\n    openapi_schema["paths"]["/api/health"]["get"].update(\n        {"tags": ["System"], "summary": "Kiểm tra trạng thái server"}\n    )\n\n    app.openapi_schema = openapi_schema\n    return app.openapi_schema\n', 'app/schemas.py': 'from __future__ import annotations\n\nimport re\nfrom typing import Any, Literal\nfrom urllib.parse import urlparse\n\nfrom pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator\n\n\nclass RecordRequest(BaseModel):\n    model_config = ConfigDict(extra="ignore", populate_by_name=True)\n\n    count: int = Field(80, ge=10, le=300)\n    size: float = Field(2.5, ge=1, le=6)\n    speed: float = Field(1.0, ge=0.2, le=3)\n    color_mode: Literal["preset", "custom"] = Field("preset", alias="colorMode")\n    color_index: int = Field(0, ge=0, le=5, alias="colorIndex")\n    custom_color: str = Field("#7fff9a", alias="customColor")\n    glow_level: Literal["low", "mid", "high"] = Field("mid", alias="glowLevel")\n    direction: Literal["up", "down", "left", "right", "up-left", "up-right", "down-left", "down-right", "random"] = "up"\n    spread: float = Field(0.4, ge=0, le=1)\n    bg_index: int = Field(1, ge=0, le=7, alias="bgIndex")\n    bg_url: str | None = Field(None, alias="bgUrl")\n    duration: int = Field(10, ge=3, le=120)\n    fps: Literal[24, 30, 60] = 60\n    width: int = Field(1920, ge=320, le=3840)\n    height: int = Field(1080, ge=240, le=2160)\n    bitrate: int = Field(5_000_000, ge=1_000_000, le=20_000_000)\n    format: Literal["webm", "mp4", "gif"] = "webm"\n    filename: str = Field("firefly", max_length=100)\n\n    @field_validator("custom_color")\n    @classmethod\n    def validate_custom_color(cls, value: str) -> str:\n        if not re.fullmatch(r"#[0-9a-fA-F]{6}", value):\n            raise ValueError("custom_color_pattern")\n        return value\n\n    @field_validator("filename")\n    @classmethod\n    def validate_filename(cls, value: str) -> str:\n        if not re.fullmatch(r"[a-zA-Z0-9_-]+", value):\n            raise ValueError("filename_pattern")\n        return value\n\n    @field_validator("bg_url", mode="before")\n    @classmethod\n    def validate_bg_url(cls, value: Any) -> str | None:\n        if value is None or value == "":\n            return None\n        if not isinstance(value, str):\n            raise ValueError("bg_url_uri")\n        parsed = urlparse(value)\n        if not parsed.scheme:\n            raise ValueError("bg_url_uri")\n        return value\n\n\nRECORD_MESSAGES = {\n    "count": {\n        "greater_than_equal": "Số lượng đom đóm phải >= 10",\n        "less_than_equal": "Số lượng đom đóm phải <= 300",\n    },\n    "size": {\n        "greater_than_equal": "Kích thước phải >= 1",\n        "less_than_equal": "Kích thước phải <= 6",\n    },\n    "speed": {\n        "greater_than_equal": "Tốc độ phải >= 0.2",\n        "less_than_equal": "Tốc độ phải <= 3.0",\n    },\n    "colorMode": {"literal_error": \'colorMode phải là "preset" hoặc "custom"\'},\n    "colorIndex": {"less_than_equal": "colorIndex phải từ 0 đến 5"},\n    "customColor": {"value_error": "customColor phải là mã hex hợp lệ (VD: #7fff9a)"},\n    "glowLevel": {"literal_error": \'glowLevel phải là "low", "mid" hoặc "high"\'},\n    "direction": {"literal_error": "direction không hợp lệ"},\n    "spread": {\n        "greater_than_equal": "Độ tản mạn phải >= 0",\n        "less_than_equal": "Độ tản mạn phải <= 1",\n    },\n    "bgIndex": {"less_than_equal": "bgIndex phải từ 0 đến 7"},\n    "bgUrl": {"value_error": "bgUrl phải là URL hợp lệ"},\n    "duration": {\n        "greater_than_equal": "Thời lượng phải >= 3 giây",\n        "less_than_equal": "Thời lượng phải <= 120 giây",\n    },\n    "fps": {"literal_error": "FPS phải là 24, 30 hoặc 60"},\n    "width": {\n        "greater_than_equal": "Chiều rộng phải >= 320px",\n        "less_than_equal": "Chiều rộng phải <= 3840px",\n    },\n    "height": {\n        "greater_than_equal": "Chiều cao phải >= 240px",\n        "less_than_equal": "Chiều cao phải <= 2160px",\n    },\n    "bitrate": {\n        "greater_than_equal": "Bitrate phải >= 1,000,000 bps",\n        "less_than_equal": "Bitrate phải <= 20,000,000 bps",\n    },\n    "format": {"literal_error": \'Format phải là "webm", "mp4" hoặc "gif"\'},\n    "filename": {\n        "value_error": "Tên file chỉ được chứa chữ cái, số, dấu gạch ngang và gạch dưới",\n        "string_too_long": "Tên file tối đa 100 ký tự",\n    },\n}\n\ndef _field_from_error(error: dict[str, Any]) -> str:\n    loc = [str(part) for part in error.get("loc", []) if part != "body"]\n    return loc[-1] if loc else ""\n\n\ndef format_record_errors(exc: ValidationError) -> list[dict[str, str]]:\n    details = []\n    for error in exc.errors():\n        field = _field_from_error(error)\n        error_type = error.get("type", "")\n        message = RECORD_MESSAGES.get(field, {}).get(error_type, error.get("msg", "Invalid value"))\n        details.append({"field": field, "message": message})\n    return details\n', 'app/main.py': 'from __future__ import annotations\n\nimport time\nfrom contextlib import asynccontextmanager\n\nfrom fastapi import FastAPI, Request\nfrom fastapi.exceptions import RequestValidationError\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.staticfiles import StaticFiles\nfrom starlette.responses import JSONResponse\n\nfrom app.api.record import router as record_router\nfrom app.config import SERVER_CONFIG\nfrom app.openapi import build_custom_openapi\nfrom app.services.renderer import close_browser\n\n\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    SERVER_CONFIG.temp_dir.mkdir(parents=True, exist_ok=True)\n    try:\n        yield\n    finally:\n        await close_browser()\n        cleaned = 0\n        for path in SERVER_CONFIG.temp_dir.iterdir():\n            if path.is_file():\n                path.unlink(missing_ok=True)\n                cleaned += 1\n        print(f"[Server] Cleaned up {cleaned} temp files.")\n\n\napp = FastAPI(\n    title="Web2Media Service",\n    version="1.0.0",\n    description="Server-side API để tạo video animation.",\n    docs_url="/docs",\n    redoc_url=None,\n    lifespan=lifespan,\n)\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n\n@app.middleware("http")\nasync def request_logger(request: Request, call_next):\n    started = time.perf_counter()\n    response = await call_next(request)\n    duration_ms = round((time.perf_counter() - started) * 1000)\n    icon = "->" if response.status_code < 400 else "x"\n    print(f"{icon} {request.method} {request.url.path} - {response.status_code} ({duration_ms}ms)")\n    return response\n\n\n@app.exception_handler(RequestValidationError)\nasync def request_validation_handler(request: Request, exc: RequestValidationError):\n    if any(error.get("type") == "json_invalid" for error in exc.errors()):\n        return JSONResponse(\n            status_code=400,\n            content={\n                "success": False,\n                "error": "JSON không hợp lệ. Kiểm tra lại cú pháp (dấu phẩy thừa, thiếu ngoặc kép...).",\n            },\n        )\n\n    return JSONResponse(status_code=422, content={"detail": exc.errors()})\n\n\napp.mount("/public", StaticFiles(directory=str(SERVER_CONFIG.public_dir)), name="public")\napp.include_router(record_router)\n\n\ndef custom_openapi():\n    return build_custom_openapi(app)\n\n\napp.openapi = custom_openapi\n\n\n@app.get("/")\nasync def root():\n    return {\n        "name": "Web2Media Service",\n        "version": "1.0.0",\n        "description": "Server-side API để tạo video animation",\n        "documentation": f"http://localhost:{SERVER_CONFIG.port}/docs",\n        "endpoints": {\n            "POST /api/record": "Tạo video với cấu hình tùy chỉnh",\n            "GET /api/presets": "Danh sách preset có sẵn",\n            "GET /api/health": "Kiểm tra trạng thái server",\n            "GET /docs": "Swagger UI - Interactive API documentation",\n        },\n    }\n\n\n@app.exception_handler(404)\nasync def not_found_handler(request: Request, exc):\n    return JSONResponse(\n        status_code=404,\n        content={"success": False, "error": f"Không tìm thấy: {request.method} {request.url.path}"},\n    )\n', 'app/api/__init__.py': '"""API route modules."""\n', 'app/api/record.py': 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\n\nfrom fastapi import APIRouter, Body\nfrom pydantic import ValidationError\nfrom starlette.background import BackgroundTask\nfrom starlette.responses import FileResponse, JSONResponse\n\nfrom app.config import BG_PRESETS, COLOR_PRESETS, DIRECTIONS, GLOW_LEVELS\nfrom app.schemas import RecordRequest, format_record_errors\nfrom app.services.converter import convert, get_mime_type\nfrom app.services.renderer import get_active_count, render_video\n\n\nrouter = APIRouter(prefix="/api")\n\n\ndef _cleanup_file(path: Path) -> None:\n    path.unlink(missing_ok=True)\n\n\n@router.post("/record")\nasync def record_video(payload: dict[str, Any] | None = Body(default_factory=dict)):\n    try:\n        params = RecordRequest.model_validate(payload or {})\n    except ValidationError as exc:\n        return JSONResponse(\n            status_code=400,\n            content={\n                "success": False,\n                "error": "Tham số không hợp lệ",\n                "details": format_record_errors(exc),\n            },\n        )\n\n    webm_path: Path | None = None\n    output_path: Path | None = None\n\n    try:\n        webm_path = await render_video(params)\n        output_path = await convert(\n            webm_path,\n            params.format,\n            bitrate=params.bitrate,\n            fps=params.fps,\n            width=params.width,\n        )\n        filename = f"{params.filename}.{params.format}"\n        return FileResponse(\n            output_path,\n            media_type=get_mime_type(params.format),\n            filename=filename,\n            background=BackgroundTask(_cleanup_file, output_path),\n        )\n    except Exception as exc:\n        if output_path:\n            output_path.unlink(missing_ok=True)\n        elif webm_path:\n            webm_path.unlink(missing_ok=True)\n\n        error = str(exc) or "Lỗi không xác định khi tạo video"\n        status_code = 429 if "giới hạn" in error else 500\n        return JSONResponse(status_code=status_code, content={"success": False, "error": error})\n\n\n@router.get("/presets")\nasync def get_presets():\n    return {\n        "success": True,\n        "data": {\n            "backgrounds": [{"index": item["index"], "label": item["label"]} for item in BG_PRESETS],\n            "colors": [\n                {"index": item["index"], "name": item["name"], "hex": item["hex"]}\n                for item in COLOR_PRESETS\n            ],\n            "directions": DIRECTIONS,\n            "glowLevels": GLOW_LEVELS,\n            "fpsOptions": [24, 30, 60],\n            "formatOptions": ["webm", "mp4", "gif"],\n        },\n    }\n\n\n@router.get("/health")\nasync def health_check():\n    from datetime import datetime, timezone\n\n    return {\n        "success": True,\n        "status": "ok",\n        "version": "1.0.0",\n        "activeRecordings": get_active_count(),\n        "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),\n    }\n', 'app/services/__init__.py': '"""Rendering and conversion services."""\n', 'app/services/converter.py': 'from __future__ import annotations\n\nimport asyncio\nimport shutil\nfrom pathlib import Path\n\n\nMIME_TYPES = {\n    "webm": "video/webm",\n    "mp4": "video/mp4",\n    "gif": "image/gif",\n}\n\n\ndef get_mime_type(format_name: str) -> str:\n    return MIME_TYPES.get(format_name, "application/octet-stream")\n\n\ndef _ffmpeg_bin() -> str:\n    ffmpeg = shutil.which("ffmpeg")\n    if not ffmpeg:\n        raise RuntimeError("Không tìm thấy ffmpeg trong PATH")\n    return ffmpeg\n\n\nasync def _run_ffmpeg(args: list[str], error_prefix: str) -> None:\n    process = await asyncio.create_subprocess_exec(\n        _ffmpeg_bin(),\n        *args,\n        stdout=asyncio.subprocess.PIPE,\n        stderr=asyncio.subprocess.PIPE,\n    )\n    _, stderr = await process.communicate()\n\n    if process.returncode != 0:\n        message = stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"{error_prefix}: {message}")\n\n\nasync def convert_to_mp4(input_path: Path, output_path: Path, bitrate: int = 5_000_000, fps: int = 60) -> Path:\n    bitrate_kbps = round(bitrate / 1000)\n    await _run_ffmpeg(\n        [\n            "-y",\n            "-i",\n            str(input_path),\n            "-c:v",\n            "libx264",\n            "-b:v",\n            f"{bitrate_kbps}k",\n            "-pix_fmt",\n            "yuv420p",\n            "-movflags",\n            "+faststart",\n            "-preset",\n            "fast",\n            "-r",\n            str(fps),\n            str(output_path),\n        ],\n        "Lỗi chuyển đổi MP4",\n    )\n    return output_path\n\n\nasync def convert_to_gif(input_path: Path, output_path: Path, fps: int = 15, width: int = 640) -> Path:\n    gif_fps = min(fps, 15)\n    gif_width = min(width, 800)\n    palette_path = input_path.with_name(f"{input_path.stem}_palette.png")\n\n    try:\n        await _run_ffmpeg(\n            [\n                "-y",\n                "-i",\n                str(input_path),\n                "-vf",\n                f"fps={gif_fps},scale={gif_width}:-1:flags=lanczos,palettegen=stats_mode=diff",\n                str(palette_path),\n            ],\n            "Lỗi tạo palette GIF",\n        )\n        await _run_ffmpeg(\n            [\n                "-y",\n                "-i",\n                str(input_path),\n                "-i",\n                str(palette_path),\n                "-filter_complex",\n                f"fps={gif_fps},scale={gif_width}:-1:flags=lanczos[x];[x][1:v]paletteuse=dither=bayer:bayer_scale=5",\n                str(output_path),\n            ],\n            "Lỗi chuyển đổi GIF",\n        )\n    finally:\n        palette_path.unlink(missing_ok=True)\n\n    return output_path\n\n\nasync def convert(webm_path: Path, format_name: str, *, bitrate: int, fps: int, width: int) -> Path:\n    if format_name == "webm":\n        return webm_path\n\n    output_path = webm_path.with_suffix(f".{format_name}")\n\n    if format_name == "mp4":\n        await convert_to_mp4(webm_path, output_path, bitrate=bitrate, fps=fps)\n    elif format_name == "gif":\n        await convert_to_gif(webm_path, output_path, fps=fps, width=width)\n    else:\n        raise RuntimeError(f"Format không được hỗ trợ: {format_name}")\n\n    webm_path.unlink(missing_ok=True)\n    return output_path\n', 'app/services/renderer.py': 'from __future__ import annotations\n\nimport asyncio\nimport base64\nimport uuid\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nimport httpx\nfrom playwright.async_api import Browser, Page, Playwright, async_playwright\n\nfrom app.config import SERVER_CONFIG\nfrom app.schemas import RecordRequest\n\n\n_playwright: Playwright | None = None\n_browser: Browser | None = None\n_browser_lock = asyncio.Lock()\n_active_lock = asyncio.Lock()\n_active_recordings = 0\n\n\ndef get_active_count() -> int:\n    return _active_recordings\n\n\nasync def _on_browser_disconnected() -> None:\n    global _browser\n    _browser = None\n\n\nasync def get_browser() -> Browser:\n    global _browser, _playwright\n\n    async with _browser_lock:\n        if _browser and _browser.is_connected():\n            return _browser\n\n        if _playwright is None:\n            _playwright = await async_playwright().start()\n\n        _browser = await _playwright.chromium.launch(\n            headless=True,\n            args=[\n                "--no-sandbox",\n                "--disable-setuid-sandbox",\n                "--disable-dev-shm-usage",\n                "--disable-gpu",\n                "--no-first-run",\n                "--no-zygote",\n                "--disable-extensions",\n                "--autoplay-policy=no-user-gesture-required",\n            ],\n        )\n        _browser.on("disconnected", lambda *_: asyncio.create_task(_on_browser_disconnected()))\n        return _browser\n\n\nasync def download_image_as_data_url(url: str) -> str:\n    headers = {\n        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",\n        "Accept": "image/*,*/*",\n        "Referer": f"{urlparse(url).scheme}://{urlparse(url).netloc}",\n    }\n\n    async with httpx.AsyncClient(follow_redirects=True, timeout=15.0, headers=headers) as client:\n        response = await client.get(url)\n        response.raise_for_status()\n        content = response.content\n\n    if len(content) < 100:\n        raise RuntimeError("Downloaded file too small - likely not an image")\n\n    mime_type = response.headers.get("content-type", "image/jpeg").split(";")[0].strip()\n    encoded = base64.b64encode(content).decode("ascii")\n    return f"data:{mime_type};base64,{encoded}"\n\n\nasync def _increment_recordings() -> None:\n    global _active_recordings\n    async with _active_lock:\n        if _active_recordings >= SERVER_CONFIG.max_concurrent:\n            raise RuntimeError(\n                f"Đã đạt giới hạn {SERVER_CONFIG.max_concurrent} video đồng thời. Vui lòng thử lại sau."\n            )\n        _active_recordings += 1\n\n\nasync def _decrement_recordings() -> None:\n    global _active_recordings\n    async with _active_lock:\n        _active_recordings = max(0, _active_recordings - 1)\n\n\nasync def render_video(params: RecordRequest) -> Path:\n    await _increment_recordings()\n    page: Page | None = None\n\n    try:\n        browser = await get_browser()\n        page = await browser.new_page()\n        await page.set_viewport_size({"width": params.width, "height": params.height})\n\n        render_page_path = SERVER_CONFIG.public_dir / "firefly-render.html"\n        await page.goto(render_page_path.as_uri(), wait_until="domcontentloaded", timeout=15_000)\n        await page.wait_for_function("window.__PAGE_READY === true", timeout=10_000)\n\n        firefly_config = {\n            "bgIndex": params.bg_index,\n            "bgCustom": params.bg_url,\n            "count": params.count,\n            "size": params.size,\n            "speed": params.speed,\n            "colorMode": params.color_mode,\n            "colorIndex": params.color_index,\n            "customColor": params.custom_color,\n            "glowLevel": params.glow_level,\n            "direction": params.direction,\n            "spread": params.spread,\n        }\n\n        if params.bg_url:\n            if params.bg_url.startswith("data:"):\n                firefly_config["bgCustom"] = params.bg_url\n            elif urlparse(params.bg_url).scheme in {"http", "https"}:\n                try:\n                    firefly_config["bgCustom"] = await download_image_as_data_url(params.bg_url)\n                except Exception as exc:\n                    print(f"[Renderer] Background download failed: {exc}. Using preset instead.")\n                    firefly_config["bgCustom"] = None\n            else:\n                firefly_config["bgCustom"] = None\n\n        await page.evaluate("(cfg) => window.__applyConfig(cfg)", firefly_config)\n        await asyncio.sleep(1.5)\n\n        base64_data = await page.evaluate(\n            """({ duration, fps, bitrate }) => window.__startRecording(duration, fps, bitrate)""",\n            {"duration": params.duration, "fps": params.fps, "bitrate": params.bitrate},\n        )\n\n        SERVER_CONFIG.temp_dir.mkdir(parents=True, exist_ok=True)\n        temp_path = SERVER_CONFIG.temp_dir / f"{uuid.uuid4()}.webm"\n        temp_path.write_bytes(base64.b64decode(base64_data))\n        return temp_path\n    finally:\n        if page:\n            await page.close()\n        await _decrement_recordings()\n\n\nasync def close_browser() -> None:\n    global _browser, _playwright\n\n    if _browser:\n        try:\n            await _browser.close()\n        finally:\n            _browser = None\n\n    if _playwright:\n        try:\n            await _playwright.stop()\n        finally:\n            _playwright = None\n', 'public/firefly-render.html': '<!DOCTYPE html>\n<html lang="vi">\n\n<head>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <title>Firefly Render (Headless)</title>\n  <style>\n    * { margin: 0; padding: 0; box-sizing: border-box; }\n    body { background: #0a0f1e; overflow: hidden; }\n\n    #bg-layer {\n      position: fixed;\n      inset: 0;\n      z-index: 0;\n      background-size: cover;\n      background-position: center;\n      background-repeat: no-repeat;\n    }\n\n    #bg-overlay {\n      position: fixed;\n      inset: 0;\n      z-index: 1;\n      background: radial-gradient(ellipse at 50% 80%, rgba(0, 30, 10, 0.45) 0%, rgb(5 10 20 / 44%) 100%);\n      pointer-events: none;\n    }\n\n    #firefly-canvas {\n      position: fixed;\n      inset: 0;\n      z-index: 2;\n      pointer-events: none;\n    }\n  </style>\n</head>\n\n<body>\n  <div id="bg-layer"></div>\n  <div id="bg-overlay"></div>\n  <canvas id="firefly-canvas"></canvas>\n\n  <script>\n    // ── Presets data ──\n    const BG_PRESETS = [\n      { label: \'Rừng\', gradient: \'radial-gradient(ellipse at 30% 60%, #0d2b14 0%, #050e08 60%, #020608 100%)\' },\n      { label: \'Đêm\', gradient: \'radial-gradient(ellipse at 50% 100%, #0d1535 0%, #040810 60%, #010205 100%)\' },\n      { label: \'Hoàng hôn\', gradient: \'linear-gradient(170deg, #0d0510 0%, #2a0e20 40%, #0a0508 100%)\' },\n      { label: \'Ao hồ\', gradient: \'radial-gradient(ellipse at 50% 80%, #071e2e 0%, #030c14 50%, #010408 100%)\' },\n      { label: \'Núi\', gradient: \'linear-gradient(160deg, #070a14 0%, #111828 40%, #050710 100%)\' },\n      { label: \'Lúa\', gradient: \'radial-gradient(ellipse at 50% 90%, #1a2208 0%, #0a0e04 60%, #040602 100%)\' },\n      { label: \'Biển\', gradient: \'linear-gradient(180deg, #03060e 0%, #061224 40%, #040d1a 100%)\' },\n      { label: \'Tím\', gradient: \'radial-gradient(ellipse at 50% 100%, #1e0f35 0%, #08051a 60%, #03020c 100%)\' },\n    ];\n\n    const COLOR_PRESETS = [\n      { name: \'Xanh lá\', h: 105, s: 85, l: 65 },\n      { name: \'Vàng\', h: 55, s: 95, l: 68 },\n      { name: \'Xanh lam\', h: 195, s: 90, l: 65 },\n      { name: \'Cam\', h: 35, s: 95, l: 65 },\n      { name: \'Trắng\', h: 0, s: 0, l: 90 },\n      { name: \'Hồng\', h: 320, s: 80, l: 75 },\n    ];\n\n    const DIR_VECTORS = {\n      \'up\': { vx: 0, vy: -1 },\n      \'down\': { vx: 0, vy: 1 },\n      \'left\': { vx: -1, vy: 0 },\n      \'right\': { vx: 1, vy: 0 },\n      \'up-left\': { vx: -0.71, vy: -0.71 },\n      \'up-right\': { vx: 0.71, vy: -0.71 },\n      \'down-left\': { vx: -0.71, vy: 0.71 },\n      \'down-right\': { vx: 0.71, vy: 0.71 },\n      \'random\': { vx: 0, vy: 0 },\n    };\n\n    const GLOW_MAP = { low: 1, mid: 2, high: 4 };\n\n    // ── Config state (will be overwritten by the server-side renderer) ──\n    let config = window.__FIREFLY_CONFIG || {\n      bgIndex: 1, bgCustom: null,\n      count: 80, size: 2.5, speed: 1.0,\n      colorMode: \'preset\',\n      colorIndex: 0,\n      customColor: \'#7fff9a\',\n      glowLevel: \'mid\',\n      direction: \'up\',\n      spread: 0.4,\n    };\n\n    // ── Canvas setup ──\n    const canvas = document.getElementById(\'firefly-canvas\');\n    const ctx = canvas.getContext(\'2d\');\n    let fireflies = [], animId;\n\n    function resize() {\n      canvas.width = window.innerWidth;\n      canvas.height = window.innerHeight;\n    }\n    window.addEventListener(\'resize\', resize);\n    resize();\n\n    // ── Hex to HSL ──\n    function hexToHsl(hex) {\n      let r = parseInt(hex.slice(1, 3), 16) / 255;\n      let g = parseInt(hex.slice(3, 5), 16) / 255;\n      let b = parseInt(hex.slice(5, 7), 16) / 255;\n      const max = Math.max(r, g, b), min = Math.min(r, g, b);\n      let h, s, l = (max + min) / 2;\n      if (max === min) { h = s = 0; }\n      else {\n        const d = max - min;\n        s = l > 0.5 ? d / (2 - max - min) : d / (max + min);\n        switch (max) {\n          case r: h = ((g - b) / d + (g < b ? 6 : 0)) / 6; break;\n          case g: h = ((b - r) / d + 2) / 6; break;\n          case b: h = ((r - g) / d + 4) / 6; break;\n        }\n      }\n      return { h: h * 360, s: s * 100, l: l * 100 };\n    }\n\n    // ── Firefly class ──\n    class Firefly {\n      constructor() { this.reset(true); }\n\n      reset(init = false) {\n        const W = canvas.width, H = canvas.height;\n        const dir = config.direction;\n        const vec = DIR_VECTORS[dir];\n        const spd = (Math.random() * 0.5 + 0.4) * config.speed;\n        const spread = config.spread;\n\n        if (dir === \'random\') {\n          const angle = Math.random() * Math.PI * 2;\n          this.vx = Math.cos(angle) * spd;\n          this.vy = Math.sin(angle) * spd;\n        } else {\n          const perp = spread * (Math.random() * 2 - 1);\n          this.vx = (vec.vx + (-vec.vy) * perp) * spd;\n          this.vy = (vec.vy + (vec.vx) * perp) * spd;\n        }\n\n        if (init) {\n          this.x = Math.random() * W;\n          this.y = Math.random() * H;\n        } else {\n          this._spawnFromEdge(W, H);\n        }\n\n        this.driftAngle = Math.random() * Math.PI * 2;\n        this.driftSpeed = (Math.random() - 0.5) * 0.03;\n        this.driftAmp = Math.random() * 0.5 + 0.1;\n        this.phase = Math.random() * Math.PI * 2;\n        this.blinkSpeed = Math.random() * 0.04 + 0.015;\n        this.baseAlpha = Math.random() * 0.45 + 0.55;\n        this.size = config.size * (Math.random() * 0.6 + 0.7);\n\n        let c;\n        if (config.colorMode === \'custom\') {\n          c = hexToHsl(config.customColor);\n        } else {\n          const preset = COLOR_PRESETS[config.colorIndex];\n          c = { h: preset.h, s: preset.s, l: preset.l };\n        }\n        this.color = {\n          h: c.h + (Math.random() * 18 - 9),\n          s: Math.max(0, c.s + (Math.random() * 18 - 9)),\n          l: c.l + (Math.random() * 14 - 7),\n        };\n      }\n\n      _spawnFromEdge(W, H) {\n        const dir = config.direction;\n        if (dir === \'up\') { this.x = Math.random() * W; this.y = H + 15; }\n        else if (dir === \'down\') { this.x = Math.random() * W; this.y = -15; }\n        else if (dir === \'left\') { this.x = W + 15; this.y = Math.random() * H; }\n        else if (dir === \'right\') { this.x = -15; this.y = Math.random() * H; }\n        else if (dir === \'up-left\') { const t = Math.random(); this.x = t < 0.5 ? Math.random() * W : W + 15; this.y = t < 0.5 ? H + 15 : Math.random() * H; }\n        else if (dir === \'up-right\') { const t = Math.random(); this.x = t < 0.5 ? Math.random() * W : -15; this.y = t < 0.5 ? H + 15 : Math.random() * H; }\n        else if (dir === \'down-left\') { const t = Math.random(); this.x = t < 0.5 ? Math.random() * W : W + 15; this.y = t < 0.5 ? -15 : Math.random() * H; }\n        else if (dir === \'down-right\') { const t = Math.random(); this.x = t < 0.5 ? Math.random() * W : -15; this.y = t < 0.5 ? -15 : Math.random() * H; }\n        else { this.x = Math.random() * W; this.y = Math.random() * H; }\n      }\n\n      update() {\n        this.driftAngle += this.driftSpeed;\n        const px = -this.vy * this.driftAmp * Math.sin(this.driftAngle) * 0.3;\n        const py = this.vx * this.driftAmp * Math.sin(this.driftAngle) * 0.3;\n        this.x += this.vx + px;\n        this.y += this.vy + py;\n        this.phase += this.blinkSpeed;\n\n        const M = 25;\n        const W = canvas.width, H = canvas.height;\n        if (this.x < -M || this.x > W + M || this.y < -M || this.y > H + M) {\n          this.reset(false);\n        }\n      }\n\n      draw() {\n        const pulse = Math.sin(this.phase) * 0.5 + 0.5;\n        const alpha = this.baseAlpha * (0.35 + 0.65 * pulse);\n        const glowMul = GLOW_MAP[config.glowLevel];\n        const glow = this.size * (2 + pulse * glowMul * 3);\n        const { h, s, l } = this.color;\n\n        const grad = ctx.createRadialGradient(this.x, this.y, 0, this.x, this.y, glow);\n        grad.addColorStop(0, `hsla(${h},${s}%,${l}%,${alpha})`);\n        grad.addColorStop(0.3, `hsla(${h},${s}%,${l}%,${alpha * 0.55})`);\n        grad.addColorStop(1, `hsla(${h},${s}%,${l}%,0)`);\n        ctx.beginPath();\n        ctx.arc(this.x, this.y, glow, 0, Math.PI * 2);\n        ctx.fillStyle = grad;\n        ctx.fill();\n\n        ctx.beginPath();\n        ctx.arc(this.x, this.y, this.size * (0.45 + pulse * 0.55), 0, Math.PI * 2);\n        ctx.fillStyle = `hsla(${h},${Math.min(100, s + 25)}%,${Math.min(100, l + 20)}%,${alpha})`;\n        ctx.fill();\n      }\n    }\n\n    // ── Animation ──\n    function initFireflies() {\n      cancelAnimationFrame(animId);\n      fireflies = Array.from({ length: config.count }, () => new Firefly());\n      loop();\n    }\n\n    function loop() {\n      ctx.clearRect(0, 0, canvas.width, canvas.height);\n      for (const f of fireflies) { f.update(); f.draw(); }\n      animId = requestAnimationFrame(loop);\n    }\n\n    // ── Apply background ──\n    function applyBackground() {\n      const bg = document.getElementById(\'bg-layer\');\n      bg.style.cssText = \'\';\n      if (config.bgCustom) {\n        bg.style.backgroundImage = \'url(\' + config.bgCustom + \')\';\n        bg.style.backgroundSize = \'cover\';\n        bg.style.backgroundPosition = \'center\';\n        bg.style.backgroundRepeat = \'no-repeat\';\n      } else {\n        bg.style.background = BG_PRESETS[config.bgIndex].gradient;\n      }\n    }\n\n    // ── Exposed API for the server-side renderer ──\n\n    /**\n     * Apply external config from the server-side renderer\n     * @param {Object} cfg - Configuration object\n     */\n    window.__applyConfig = function (cfg) {\n      config = { ...config, ...cfg };\n      applyBackground();\n      initFireflies();\n    };\n\n    /**\n     * Start recording and return blob data as base64\n     * @param {number} duration - Duration in seconds\n     * @param {number} fps - Frames per second\n     * @param {number} bitrate - Video bitrate in bps\n     * @returns {Promise<string>} Base64-encoded webm video data\n     */\n    window.__startRecording = function (duration, fps, bitrate) {\n      return new Promise((resolve, reject) => {\n        const W = canvas.width, H = canvas.height;\n        const offscreen = document.createElement(\'canvas\');\n        offscreen.width = W;\n        offscreen.height = H;\n        const offCtx = offscreen.getContext(\'2d\');\n\n        function drawFrame() {\n          offCtx.clearRect(0, 0, W, H);\n          // Draw background\n          if (config.bgCustom) {\n            if (!drawFrame._img || drawFrame._imgSrc !== config.bgCustom) {\n              drawFrame._img = new Image();\n              drawFrame._img.src = config.bgCustom;\n              drawFrame._imgSrc = config.bgCustom;\n            }\n            if (drawFrame._img.complete) {\n              const iw = drawFrame._img.naturalWidth, ih = drawFrame._img.naturalHeight;\n              const scale = Math.max(W / iw, H / ih);\n              const sw = iw * scale, sh = ih * scale;\n              offCtx.drawImage(drawFrame._img, (W - sw) / 2, (H - sh) / 2, sw, sh);\n            }\n          } else {\n            const grad = BG_PRESETS[config.bgIndex].gradient;\n            offCtx.fillStyle = grad;\n            offCtx.fillRect(0, 0, W, H);\n          }\n          // Dark vignette overlay\n          const vig = offCtx.createRadialGradient(W / 2, H * 0.8, 0, W / 2, H * 0.8, W * 0.9);\n          vig.addColorStop(0, \'rgba(0,30,10,0.45)\');\n          vig.addColorStop(1, \'rgba(5,10,20,0.75)\');\n          offCtx.fillStyle = vig;\n          offCtx.fillRect(0, 0, W, H);\n          // Draw firefly canvas on top\n          offCtx.drawImage(canvas, 0, 0);\n        }\n\n        // Patched loop that also draws on offscreen\n        function patchedLoop() {\n          offCtx.clearRect(0, 0, W, H);\n          ctx.clearRect(0, 0, W, H);\n          for (const f of fireflies) { f.update(); f.draw(); }\n          drawFrame();\n          animId = requestAnimationFrame(patchedLoop);\n        }\n        cancelAnimationFrame(animId);\n        patchedLoop();\n\n        // Start MediaRecorder\n        const stream = offscreen.captureStream(fps);\n        const mimeType = MediaRecorder.isTypeSupported(\'video/webm;codecs=vp9\')\n          ? \'video/webm;codecs=vp9\' : \'video/webm\';\n        const recorder = new MediaRecorder(stream, {\n          mimeType,\n          videoBitsPerSecond: bitrate,\n        });\n        const chunks = [];\n\n        recorder.ondataavailable = (e) => {\n          if (e.data.size > 0) chunks.push(e.data);\n        };\n\n        recorder.onstop = () => {\n          const blob = new Blob(chunks, { type: mimeType });\n          // Convert blob to base64\n          const reader = new FileReader();\n          reader.onloadend = () => {\n            // Restore normal loop\n            cancelAnimationFrame(animId);\n            loop();\n            // Return base64 data (strip data URL prefix)\n            const base64 = reader.result.split(\',\')[1];\n            resolve(base64);\n          };\n          reader.onerror = () => reject(\'Failed to read recording blob\');\n          reader.readAsDataURL(blob);\n        };\n\n        recorder.onerror = (e) => reject(\'MediaRecorder error: \' + e.error);\n\n        recorder.start(200); // Collect data every 200ms\n\n        // Stop after duration\n        setTimeout(() => {\n          if (recorder.state === \'recording\') {\n            recorder.stop();\n          }\n        }, duration * 1000);\n      });\n    };\n\n    // ── Signal that page is ready ──\n    window.__PAGE_READY = false;\n\n    // ── Init ──\n    applyBackground();\n    initFireflies();\n    window.__PAGE_READY = true;\n  </script>\n</body>\n\n</html>\n', 'tests/test_api_smoke.py': 'from fastapi.testclient import TestClient\n\nfrom app.main import app\n\n\ndef test_health_and_presets_endpoints():\n    with TestClient(app) as client:\n        health = client.get("/api/health")\n        presets = client.get("/api/presets")\n\n    assert health.status_code == 200\n    assert health.json()["status"] == "ok"\n    assert presets.status_code == 200\n    assert len(presets.json()["data"]["backgrounds"]) == 8\n\n\ndef test_root_and_404_contract():\n    with TestClient(app) as client:\n        root = client.get("/")\n        missing = client.get("/missing")\n\n    assert root.status_code == 200\n    assert root.json()["name"] == "Web2Media Service"\n    assert missing.status_code == 404\n    assert missing.json()["success"] is False\n\n\ndef test_record_validation_contract():\n    with TestClient(app) as client:\n        response = client.post("/api/record", json={"count": 301})\n\n    assert response.status_code == 400\n    assert response.json() == {\n        "success": False,\n        "error": "Tham số không hợp lệ",\n        "details": [{"field": "count", "message": "Số lượng đom đóm phải <= 300"}],\n    }\n\n', 'tests/test_openapi.py': 'from fastapi.testclient import TestClient\n\nfrom app.main import app\n\n\ndef test_openapi_record_request_has_full_schema_and_examples():\n    with TestClient(app) as client:\n        schema = client.get("/openapi.json").json()\n\n    record = schema["paths"]["/api/record"]["post"]["requestBody"]["content"]["application/json"]\n    record_schema = schema["components"]["schemas"]["RecordRequest"]\n\n    assert record["schema"]["$ref"] == "#/components/schemas/RecordRequest"\n    assert "minimal" in record["examples"]\n    assert "custom_color_mp4" in record["examples"]\n    assert "count" in record_schema["properties"]\n    assert "customColor" in record_schema["properties"]\n    assert "bgUrl" in record_schema["properties"]\n    assert "format" in record_schema["properties"]\n\n', 'tests/test_schemas.py': 'from pydantic import ValidationError\n\nfrom app.schemas import RecordRequest, format_record_errors\n\n\ndef test_record_request_applies_defaults_and_ignores_unknown_fields():\n    params = RecordRequest.model_validate({"duration": 5, "unknown": "ignored"})\n\n    assert params.duration == 5\n    assert params.count == 80\n    assert params.format == "webm"\n\n\ndef test_record_request_formats_validation_errors_like_api_contract():\n    try:\n        RecordRequest.model_validate({"count": 301})\n    except ValidationError as exc:\n        details = format_record_errors(exc)\n    else:\n        raise AssertionError("Expected validation error")\n\n    assert details == [{"field": "count", "message": "Số lượng đom đóm phải <= 300"}]\n\n'}

for relative_path, content in TEXT_FILES.items():
    path = Path(relative_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

Path("temp").mkdir(parents=True, exist_ok=True)

print("Project files created in", Path.cwd())
print("FastAPI entrypoint: app.main:app")
print("Available APIs: POST /api/record, GET /api/presets, GET /api/health")


In [ ]:
# Runtime setup for Google Colab or local Jupyter/Windows.
# Run this after the first cell has recreated requirements.txt.
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path


def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.check_call([str(part) for part in cmd])

is_linux = platform.system().lower() == "linux"
has_apt = shutil.which("apt-get") is not None

if is_linux and has_apt:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"])
else:
    print("Skipping apt-get: this is not a Linux/Colab runtime.")

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "nest_asyncio"])

# Install Chromium for Playwright. On Linux/Colab, --with-deps also installs browser libraries.
playwright_cmd = [sys.executable, "-m", "playwright", "install"]
if is_linux and has_apt:
    playwright_cmd.append("--with-deps")
playwright_cmd.append("chromium")
run(playwright_cmd)

# Local Windows often has no ffmpeg in PATH. Use imageio-ffmpeg as a portable fallback.
if shutil.which("ffmpeg") is None:
    run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"])
    import imageio_ffmpeg

    ffmpeg_exe = Path(imageio_ffmpeg.get_ffmpeg_exe())
    os.environ["PATH"] = str(ffmpeg_exe.parent) + os.pathsep + os.environ.get("PATH", "")
    print("Using bundled ffmpeg:", ffmpeg_exe)
else:
    print("Using ffmpeg from PATH:", shutil.which("ffmpeg"))

print("Setup complete. Now run the server cell.")


In [ ]:
# Start the FastAPI service and expose it through Colab's built-in proxy.
import os
import sys
import time
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()
os.environ.setdefault("PORT", "4526")

try:
    colab_web2media_server.should_exit = True
    time.sleep(1)
except NameError:
    pass

# If this notebook cell is rerun after recreating files, clear cached app modules.
for module_name in list(sys.modules):
    if module_name == "app" or module_name.startswith("app."):
        del sys.modules[module_name]

config = uvicorn.Config("app.main:app", host="0.0.0.0", port=4526, log_level="info")
colab_web2media_server = uvicorn.Server(config)
thread = threading.Thread(target=colab_web2media_server.run, daemon=True)
thread.start()
time.sleep(3)

try:
    from google.colab import output
    base_url = output.eval_js("google.colab.kernel.proxyPort(4526)")
    if not base_url.endswith("/"):
        base_url += "/"
    print("Service URL:", base_url)
    print("Docs URL:   ", base_url + "docs")
    output.serve_kernel_port_as_window(4526)
except Exception:
    print("Service URL: http://127.0.0.1:4526")
    print("Docs URL:    http://127.0.0.1:4526/docs")


In [ ]:
# Quick smoke check. Run this after the server cell is ready.
import httpx

base_url = "http://127.0.0.1:4526"
health = httpx.get(base_url + "/api/health", timeout=30).json()
presets = httpx.get(base_url + "/api/presets", timeout=30).json()
openapi = httpx.get(base_url + "/openapi.json", timeout=30).json()
record_schema = openapi["components"]["schemas"]["RecordRequest"]
record_examples = openapi["paths"]["/api/record"]["post"]["requestBody"]["content"]["application/json"]["examples"]
print("Health:", health)
print("Background presets:", len(presets["data"]["backgrounds"]))
print("Swagger record params:", ", ".join(record_schema["properties"].keys()))
print("Swagger record examples:", ", ".join(record_examples.keys()))


In [ ]:
# Optional: generate a tiny 3-second WebM video to verify Playwright rendering.
# The output will be saved as smoke.webm.
import httpx
from pathlib import Path

payload = {
    "duration": 3,
    "width": 320,
    "height": 240,
    "fps": 24,
    "count": 10,
    "format": "webm",
    "filename": "smoke",
}
response = httpx.post("http://127.0.0.1:4526/api/record", json=payload, timeout=180)
response.raise_for_status()
Path("smoke.webm").write_bytes(response.content)
print("Saved", Path("smoke.webm").resolve(), "bytes=", len(response.content))


In [ ]:
# Optional: run the bundled Python tests.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt"])
subprocess.check_call([sys.executable, "-m", "pytest"])
